In [1]:
import time
import copy
import gc
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.transforms import v2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    roc_auc_score, confusion_matrix, recall_score,
)

import optuna
from joblib import Parallel, delayed

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision('high')      # usa TF32 em Ampere+
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = torch.cuda.is_available() 
print(f"Dispositivo de treinamento: {device}")

DATASET_PATH = Path(r"/home/paulo/TCC/The IQ-OTHNCCD lung cancer dataset/IQ_dataset")
RESOLUTION = 224         

N_REPETICOES = 20    
N_TRIALS = 50
EPOCAS_CALIB = 20        
EPOCAS_FINAL = 150      
PACIENCIA = 20       

BASE_MOBILENET = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

Dispositivo de treinamento: cuda


In [3]:
def load_data(dataset_path):
    filepaths, labels = [], []
    for filepath in dataset_path.rglob("*.jpg"):
        filepaths.append(str(filepath))
        labels.append(int(filepath.parent.name))
    return filepaths, labels


def balance_data(filepaths, labels, seed):
    rng = np.random.RandomState(seed)

    paths_cls0 = [p for p, l in zip(filepaths, labels) if l == 0]
    paths_cls1 = [p for p, l in zip(filepaths, labels) if l == 1]

    n_needed   = len(paths_cls1) - len(paths_cls0)
    extra = rng.choice(paths_cls0, n_needed, replace=False).tolist()
    paths_cls0 = paths_cls0 + extra

    combined = list(zip(paths_cls0 + paths_cls1, [0] * len(paths_cls0) + [1] * len(paths_cls1)))
    rng.shuffle(combined)
    bal_paths, bal_labels = zip(*combined)
    return list(bal_paths), list(bal_labels)


# Carrega todos os dados uma única vez
ALL_PATHS, ALL_LABELS = load_data(DATASET_PATH)
print(f"Total de imagens carregadas: {len(ALL_PATHS)}")
print(f"  Normais (0): {ALL_LABELS.count(0)}")
print(f"  Tumor   (1): {ALL_LABELS.count(1)}")

Total de imagens carregadas: 1097
  Normais (0): 416
  Tumor   (1): 681


In [4]:
class IQOTHDataset(Dataset):
    # Transform de CPU: redimensiona e converte para tensor float32 em [0, 1]
    _cpu_transform = v2.Compose([
        v2.Resize((RESOLUTION, RESOLUTION), antialias=True),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ])

    def __init__(self, filepaths, labels):
        self.labels = labels
        self.images = [
            self._cpu_transform(Image.open(p).convert("RGB"))
            for p in filepaths
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], torch.tensor(self.labels[idx], dtype=torch.float32)

IMAGENET_NORM = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),  # espelhamento
    v2.RandomRotation(15),            # rotação
    v2.Normalize(**IMAGENET_NORM),
])

val_transforms = v2.Compose([
    v2.Normalize(**IMAGENET_NORM),   # sem augmentation na validação/teste
])

In [5]:
def build_model(params):
    model = copy.deepcopy(BASE_MOBILENET)

    for param in model.features.parameters():
        param.requires_grad = False

    # Fine-Tuning progressiva para MobileNetV2 propriedade 'features' tem 19 módulos (0 a 18).
    unfreeze_starts = {
        0: 19, # Não descongela nada (só treina a camada densa final)
        1: 17, # Descongela os últimos 2 blocos
        2: 14, # Descongela os últimos 5 blocos
        3: 10  # Descongela quase a metade final da rede
    }
    
    start_idx = unfreeze_starts[params['fine_tune_blocks']]

    # Descongela as camadas a partir do índice mapeado
    for i in range(start_idx, len(model.features)):
        for param in model.features[i].parameters():
            param.requires_grad = True

    # 3. Substituição do Classificador
    # A MobileNetV2 entrega um vetor de 1280 features antes da classificação
    in_features = 1280
    neuronios   = [params['neurons_l1'], params['neurons_l2'], params['neurons_l3']]
    layers      = []

    for i in range(params['num_hidden_layers']):
        layers += [
            nn.Linear(in_features, neuronios[i]),
            nn.ReLU(),
            nn.Dropout(params['dropout_rate']),
        ]
        in_features = neuronios[i]

    layers.append(nn.Linear(in_features, 1))  
    
    # Substitui o classificador original pelo do Optuna
    model.classifier = nn.Sequential(*layers)

    return model.to(device)

In [6]:
def train_and_evaluate(params, is_calibration, rep_seed, data_splits):
    X_train_base, X_val, y_train_base, y_val = data_splits

    X_train, y_train = balance_data(X_train_base, y_train_base, seed=rep_seed)

    max_epochs = EPOCAS_CALIB if is_calibration else EPOCAS_FINAL

    train_loader = DataLoader(IQOTHDataset(X_train, y_train),
                              batch_size=params['batch_size'], shuffle=True,
                              num_workers=0, pin_memory=True)
    val_loader   = DataLoader(IQOTHDataset(X_val, y_val),
                              batch_size=params['batch_size'], shuffle=False,
                              num_workers=0, pin_memory=True)

    model     = build_model(params)
    optimizer = optim.Adam(model.parameters(), lr=params['lr'],
                           weight_decay=params['weight_decay'])
    criterion = nn.BCEWithLogitsLoss()

    use_amp = device.type == 'cuda'
    scaler  = torch.amp.GradScaler('cuda') if use_amp else None

    torch.manual_seed(rep_seed)
    if use_amp:
        torch.cuda.manual_seed(rep_seed)
    best_f1 = 0.0
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0
    last_epoch = 0

    for epoch in range(max_epochs):
        last_epoch = epoch + 1

        model.train()
        for imgs, lbls in train_loader:
            imgs = train_transforms(imgs.to(device, non_blocking=True))
            lbls = lbls.to(device, non_blocking=True).unsqueeze(1)

            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.amp.autocast('cuda'):
                    loss = criterion(model(imgs), lbls)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = criterion(model(imgs), lbls)
                loss.backward()
                optimizer.step()

        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs  = val_transforms(imgs.to(device, non_blocking=True))
                preds = (torch.sigmoid(model(imgs)) >= 0.5).flatten().cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(lbls.numpy())

        val_f1 = f1_score(all_labels, all_preds, zero_division=0)

        if val_f1 > best_f1:
            best_f1          = val_f1
            best_weights     = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PACIENCIA:
                break

    # Libera VRAM ao terminar
    del model, optimizer, criterion, scaler, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if is_calibration:
        return best_f1  # Optuna maximiza este valor

    # Modo final: reconstrói o modelo e carrega os melhores pesos
    final_model = build_model(params)
    final_model.load_state_dict(best_weights)
    return final_model, last_epoch

In [7]:
def executar_repeticao(rep):

    optuna.logging.set_verbosity(optuna.logging.ERROR) 
    print(f"[Iniciando] Repetição {rep}...")

    X_train_base, X_temp, y_train_base, y_temp = train_test_split(
        ALL_PATHS, ALL_LABELS, test_size=0.30, random_state=rep, stratify=ALL_LABELS
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=rep, stratify=y_temp
    )
    data_splits = (X_train_base, X_val, y_train_base, y_val)

    def objective(trial):
        params = {
            'lr':                trial.suggest_float('lr', 1e-5, 1e-2, log=True),
            'batch_size':        trial.suggest_categorical('batch_size', [64, 128]),
            'weight_decay':      trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
            'num_hidden_layers': trial.suggest_int('num_hidden_layers', 1, 3),
            'neurons_l1':        trial.suggest_int('neurons_l1', 64, 512, step=64),
            'neurons_l2':        trial.suggest_int('neurons_l2', 64, 512, step=64),
            'neurons_l3':        trial.suggest_int('neurons_l3', 64, 512, step=64),
            'dropout_rate':      trial.suggest_float('dropout_rate', 0.2, 0.5),
            'fine_tune_blocks':  trial.suggest_int('fine_tune_blocks', 0, 3),
        }
        return train_and_evaluate(params, is_calibration=True, rep_seed=rep, data_splits=data_splits)

    t0_calib = time.time()
    sampler = optuna.samplers.GPSampler(seed=rep)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=N_TRIALS)
    t_calib = time.time() - t0_calib

    best_params = study.best_params

    t0_treino = time.time()
    final_model, n_epochs = train_and_evaluate(
        best_params, is_calibration=False, rep_seed=rep, data_splits=data_splits
    )
    t_treino = time.time() - t0_treino

    final_model.eval()
    test_loader = DataLoader(
        IQOTHDataset(X_test, y_test),
        batch_size=best_params['batch_size'], shuffle=False,
        num_workers=0, pin_memory=True
    )

    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs  = val_transforms(imgs.to(device, non_blocking=True))
            probs = torch.sigmoid(final_model(imgs)).flatten().cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend((probs >= 0.5).astype(int))
            all_labels.extend(lbls.numpy())

    metrics = {
        'acc': accuracy_score(all_labels, all_preds),
        'recall': recall_score(all_labels, all_preds, zero_division=0),
        'f1': f1_score(all_labels, all_preds, zero_division=0),
        'auc': roc_auc_score(all_labels, all_probs),
    }
    cm = confusion_matrix(all_labels, all_preds)

    torch.save(final_model.state_dict(), f"MobileNetV2_iter_{rep}.pth")

    with open(f"MobileNetV2_iter_{rep}_eval.txt", "w") as f:
        f.write(f"=== Resultados da Repetição {rep} (MobileNetV2) ===\n\n")
        f.write("--- CUSTO COMPUTACIONAL ---\n")
        f.write(f"Calibração : {t_calib:.2f} s ({t_calib/60:.2f} min)\n")
        f.write(f"Treinamento: {t_treino:.2f} s ({t_treino/60:.2f} min)\n")
        f.write(f"Épocas     : {n_epochs}\n")
        f.write(f"Total      : {(t_calib + t_treino)/60:.2f} min\n\n")
        f.write("--- MELHORES HIPERPARÂMETROS ---\n")
        for k, v in best_params.items():
            f.write(f"  {k}: {v}\n")
        f.write("\n--- MÉTRICAS DE TESTE ---\n")
        for k, v in metrics.items():
            f.write(f"  {k.upper():>8}: {v:.4f}\n")
        f.write(f"\nMatriz de Confusão:\n{cm}\n")

    del final_model, test_loader, study, sampler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"[Concluída] Repetição {rep} | F1={metrics['f1']:.4f}")
    return {'rep': rep, 't_calib': t_calib, 't_treino': t_treino, **metrics}

In [8]:
if __name__ == "__main__":
    print(f"Iniciando {N_REPETICOES} repetições em paralelo (4 workers)...\n")

    tempo_init = time.time()

    all_results = Parallel(n_jobs=4, verbose=10)(
        delayed(executar_repeticao)(rep) for rep in range(1, N_REPETICOES + 1)
    )

    t_real = time.time() - tempo_init
    print("\nTodas as repetições concluídas com sucesso!")

Iniciando 20 repetições em paralelo (4 workers)...



[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


[Iniciando] Repetição 1...
[Iniciando] Repetição 2...
[Iniciando] Repetição 3...
[Iniciando] Repetição 4...


/tmp/ipykernel_438362/227847388.py:29: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
/tmp/ipykernel_438362/227847388.py:29: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
/tmp/ipykernel_438362/227847388.py:29: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
/tmp/ipykernel_438362/227847388.py:29: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.


[Concluída] Repetição 1 | F1=0.9053
[Iniciando] Repetição 5...


...

/tmp/ipykernel_438362/227847388.py:29: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.


[Concluída] Repetição 16 | F1=0.9505
[Iniciando] Repetição 20...


[Parallel(n_jobs=4)]: Done  16 out of  20 | elapsed: 98.5min remaining: 24.6min
/tmp/ipykernel_438362/227847388.py:29: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.


[Concluída] Repetição 17 | F1=0.8584
[Concluída] Repetição 19 | F1=0.8756
[Concluída] Repetição 18 | F1=0.9146
[Concluída] Repetição 20 | F1=0.9238

Todas as repetições concluídas com sucesso!


[Parallel(n_jobs=4)]: Done  20 out of  20 | elapsed: 121.3min finished


In [9]:
metric_names = ['acc', 'recall', 'f1', 'auc']

report  = "RESULTADO FINAL - MobileNetV2\n"
report += "MÉTRICAS DE CLASSIFICAÇÃO (Média ± Desvio Padrão):\n"

for m in metric_names:
    valores = [r[m] for r in all_results]
    report += f"  {m.upper():>8}: {np.mean(valores):.4f} ± {np.std(valores):.4f}\n"

t_calib_total_maquina  = sum(r['t_calib']  for r in all_results)
t_treino_total_maquina = sum(r['t_treino'] for r in all_results)
t_total_maquina        = t_calib_total_maquina + t_treino_total_maquina

n_jobs = 4 

report += "CUSTO COMPUTACIONAL:\n\n"

report += "1. TEMPO DE MÁQUINA:\n"
report += f"  Calibração (Optuna): {t_calib_total_maquina/3600:.2f} h ({t_calib_total_maquina/60:.2f} min)\n"
report += f"  Treinamento Final  : {t_treino_total_maquina/3600:.2f} h ({t_treino_total_maquina/60:.2f} min)\n"
report += f"  Total da Máquina   : {t_total_maquina/3600:.2f} h\n\n"

report += "2. TEMPO REAL (média):\n"
report += f"  Calibração         : {(t_calib_total_maquina / n_jobs)/3600:.2f} h ({(t_calib_total_maquina / n_jobs)/60:.2f} min)\n"
report += f"  Treinamento        : {(t_treino_total_maquina / n_jobs)/3600:.2f} h ({(t_treino_total_maquina / n_jobs)/60:.2f} min)\n"

report += f"  TEMPO DOS 2 JUNTOS : {t_real/3600:.2f} h ({t_real/60:.2f} min)\n"

print(report)

with open("MobileNetV2_Resultado.txt", "w", encoding="utf-8") as f:
    f.write(report)

print("\nRelatório salvo como 'MobileNetV2_Resultado.txt'")

RESULTADO FINAL - MobileNetV2
MÉTRICAS DE CLASSIFICAÇÃO (Média ± Desvio Padrão):
       ACC: 0.8830 ± 0.0307
    RECALL: 0.8767 ± 0.0510
        F1: 0.9031 ± 0.0250
       AUC: 0.9505 ± 0.0279
CUSTO COMPUTACIONAL:

1. TEMPO DE MÁQUINA:
  Calibração (Optuna): 7.70 h (462.18 min)
  Treinamento Final  : 0.34 h (20.44 min)
  Total da Máquina   : 8.04 h

2. TEMPO REAL (média):
  Calibração         : 1.93 h (115.54 min)
  Treinamento        : 0.09 h (5.11 min)
  TEMPO DOS 2 JUNTOS : 2.02 h (121.32 min)


Relatório salvo como 'MobileNetV2_Resultado.txt'
